# K2 NDVI phenology at field plots — first spring maximum → subsequent minimum

Samples **plot locations only** from annual K2 intercept–amplitude–phase GeoTIFFs (no wall-to-wall processing or exports). Selects the **first qualifying local NDVI maximum** in a prespecified spring window, followed by its **first subsequent local minimum**. Measures timing, peak-to-trough decline, decline duration, and rate. Flags missing/weak extrema; shows fitted curves with marked events and distribution plots.

**Caution:** The phase convention and time origin must match the Earth Engine harmonic export. Verify against observed NDVI at several plots. No phenology classifier is fitted here. A periodic K2 curve is a smooth approximation, not an observation of plant phenology.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from pyproj import Transformer
from IPython.display import display

HARMONIC_ROOT = Path(r"A:\NCA_DATA\S2_Harmonics")
POINTS_CSV = Path(r"A:\NCA_DATA\Validation\Supervised_State_Classification\FieldVeg_K2_IA45_Training.csv")
YEARS = list(range(2016, 2027))  # 2026 included automatically once data exist
# Field table must contain plot coordinates; update names below after inspecting printed columns.
PLOT_ID_COL = 'PlotID'
YEAR_COL = 'Year'
X_COL = 'Easting'
Y_COL = 'Northing'
POINT_CRS = 'EPSG:26911'  # update if source coordinates are lon/lat (EPSG:4326)
# IMPORTANT: inspect the TIFF band names and confirm how phase was calculated.
# For y = c + A cos(wt - phi), choose 'cos_minus'.
# For y = c + A cos(wt + phi), choose 'cos_plus'.
# For y = c + A sin(wt + phi), choose 'sin_plus'.
PHASE_CONVENTION = 'cos_minus'
PHASE_UNITS = 'radians'  # or 'degrees'
TIME_ORIGIN_DOY = 1  # must agree with export's Fourier time origin
# Predeclared phenological windows; evaluate sensitivity before selecting any final features.
SPRING_PEAK_WINDOW = (30, 180)     # DOY inclusive; candidate early-season peaks
SUMMER_TROUGH_END_DOY = 300        # no minimum after this day
MAX_DECLINE_DAYS = 180             # require subsequent trough within 180 days
MIN_PEAK_PROMINENCE_FRACTION = 0.10 # relative to full fitted annual NDVI range
MIN_RANGE = 0.025                 # QC threshold in NDVI; not ecological significance
DOY = np.arange(1, 366)


In [ ]:
points = pd.read_csv(POINTS_CSV, low_memory=False)
print('Columns:', list(points.columns))
required = [PLOT_ID_COL, YEAR_COL, X_COL, Y_COL]
missing = [c for c in required if c not in points.columns]
if missing:
    raise KeyError(f'Missing plot columns {missing}. Set X_COL, Y_COL, POINT_CRS above based on printed columns.')
points = points.dropna(subset=required).copy()
points[YEAR_COL] = pd.to_numeric(points[YEAR_COL], errors='coerce')
points = points[points[YEAR_COL].isin(YEARS)].copy()
points = points.drop_duplicates(subset=[PLOT_ID_COL, YEAR_COL])
print('Plot-years:', len(points)); display(points[required].head())


In [ ]:
def annual_path(year):
    return HARMONIC_ROOT / str(year) / f'{year}_K2_InterceptAmpPhase_nobs_RMSE.tif'

def band_names(src):
    return [str(x or f'band_{i}').strip() for i, x in enumerate(src.descriptions, 1)]

for yr in YEARS:
    path = annual_path(yr)
    if path.exists():
        with rasterio.open(path) as ds:
            print('Example TIFF:', path, '| CRS:', ds.crs)
            print('NDVI-related bands:', [(i, s) for i,s in enumerate(band_names(ds),1) if 'ndvi' in s.lower()])
        break
else:
    raise FileNotFoundError('No annual K2 TIFF found; verify HARMONIC_ROOT.')


In [ ]:
def resolve_ndvi_bands(src):
    # Requires explicit NDVI intercept, amplitude 1/2 and phase 1/2.
    # Prevents silently using phase from a different index/order.
    names = band_names(src)
    groups = {
        'beta0': [r'^(constant|intercept|beta0)[_-]ndvi$', r'^ndvi[_-](constant|intercept|beta0)$'],
        'a1': [r'^(amp1|amplitude1|amp_1)[_-]ndvi$', r'^ndvi[_-](amp1|amplitude1|amp_1)$'],
        'a2': [r'^(amp2|amplitude2|amp_2)[_-]ndvi$', r'^ndvi[_-](amp2|amplitude2|amp_2)$'],
        'p1': [r'^(phase1|phase_1|phi1|phi_1)[_-]ndvi$', r'^ndvi[_-](phase1|phase_1|phi1|phi_1)$'],
        'p2': [r'^(phase2|phase_2|phi2|phi_2)[_-]ndvi$', r'^ndvi[_-](phase2|phase_2|phi2|phi_2)$'],
    }
    idx = {}
    for key, patterns in groups.items():
        match = [i for i,name in enumerate(names, 1) if any(re.fullmatch(pt,name.lower()) for pt in patterns)]
        if len(match) != 1:
            raise ValueError(f'{key}: expected one NDVI band, got {match}. Available NDVI bands: '
                             +repr([(i,s) for i,s in enumerate(names,1) if 'ndvi' in s.lower()]))
        idx[key] = match[0]
    return idx

def sample_year(df, year):
    path=annual_path(year)
    if not path.exists():
        print(f'Skipping {year}: no raster')
        return pd.DataFrame()
    with rasterio.open(path) as ds:
        ix=resolve_ndvi_bands(ds)
        xy=df[[X_COL,Y_COL]].apply(pd.to_numeric,errors='coerce').to_numpy(float)
        if not np.isfinite(xy).all():
            raise ValueError(f'Invalid plot coordinates for {year}')
        if str(ds.crs) != str(POINT_CRS):
            t=Transformer.from_crs(POINT_CRS,ds.crs,always_xy=True)
            xy=np.array([t.transform(x,y) for x,y in xy])
        coords=list(map(tuple,xy))
        # Only 5 bands at selected points; never reads complete raster.
        vals=np.asarray(list(ds.sample(coords,indexes=list(ix.values()),masked=True)))
        values=np.ma.filled(vals.astype(float),np.nan)
        result=df[[PLOT_ID_COL,YEAR_COL]].copy().reset_index(drop=True)
        for j,key in enumerate(ix): result[key]=values[:,j]
        result['in_bounds']=[ds.bounds.left<=x<=ds.bounds.right and ds.bounds.bottom<=y<=ds.bounds.top for x,y in coords]
        result.loc[~result['in_bounds'],list(ix)] = np.nan
        return result

frames=[]
for year in YEARS:
    sub=points[points[YEAR_COL].eq(year)]
    if len(sub):
        sampled=sample_year(sub,year)
        if len(sampled): frames.append(sampled)
if not frames: raise RuntimeError('No plot-year samples available.')
sampled=pd.concat(frames,ignore_index=True)
valid=np.isfinite(sampled[['beta0','a1','a2','p1','p2']]).all(axis=1)
print(f'Complete phase/amp fits: {valid.sum()} / {len(sampled)}')
display(sampled.head())


In [ ]:
def reconstruct_ndvi(rec, doy=DOY):
    t = np.asarray(doy, dtype=float) - TIME_ORIGIN_DOY
    theta = 2 * np.pi * t / 365.0
    p1, p2 = float(rec['p1']), float(rec['p2'])
    if PHASE_UNITS == 'degrees':
        p1, p2 = np.deg2rad([p1, p2])
    if PHASE_CONVENTION == 'cos_minus':
        v = rec['beta0'] + rec['a1'] * np.cos(theta - p1) + rec['a2'] * np.cos(2 * theta - p2)
    elif PHASE_CONVENTION == 'cos_plus':
        v = rec['beta0'] + rec['a1'] * np.cos(theta + p1) + rec['a2'] * np.cos(2 * theta + p2)
    elif PHASE_CONVENTION == 'sin_plus':
        v = rec['beta0'] + rec['a1'] * np.sin(theta + p1) + rec['a2'] * np.sin(2 * theta + p2)
    else:
        raise ValueError('Unknown PHASE_CONVENTION')
    return np.asarray(v, dtype=float)


def first_spring_peak_and_trough(v):
    """Find first meaningful spring local maximum and next summer local minimum.

    Extrema use circular neighbors (Fourier curve is periodic), but their selection
    uses explicit calendar-day windows. First qualifying candidate is selected.
    Qualifying means the drop to the NEXT local minimum exceeds a prespecified
    fraction of annual fitted range. This avoids selecting insignificant ripples.
    """
    v = np.asarray(v, dtype=float)
    if v.shape != (365,) or not np.isfinite(v).all():
        return None
    annual_range = float(np.ptp(v))
    if annual_range < MIN_RANGE:
        return None
    maxima = np.flatnonzero((v > np.roll(v, 1)) & (v >= np.roll(v, -1)))
    minima = np.flatnonzero((v < np.roll(v, 1)) & (v <= np.roll(v, -1)))
    for peak in maxima:  # sorted earliest DOY first
        peak_doy = int(peak + 1)
        if not SPRING_PEAK_WINDOW[0] <= peak_doy <= SPRING_PEAK_WINDOW[1]:
            continue
        next_minima = [i for i in minima if
                       peak < i and (i - peak) <= MAX_DECLINE_DAYS
                       and (i + 1) <= SUMMER_TROUGH_END_DOY]
        if not next_minima:
            continue
        trough = int(next_minima[0])
        decline = float(v[peak] - v[trough])
        if decline >= MIN_PEAK_PROMINENCE_FRACTION * annual_range:
            return int(peak), trough
    return None


def phenology_metrics(rec):
    v = reconstruct_ndvi(rec)
    annual_range = float(np.ptp(v))
    event = first_spring_peak_and_trough(v)
    out = {
        'spring_peak_doy': np.nan,
        'summer_min_doy': np.nan,
        'spring_peak_ndvi_fit': np.nan,
        'summer_min_ndvi_fit': np.nan,
        'decline_duration_days': np.nan,
        'decline_magnitude_ndvi': np.nan,
        'mean_decline_rate_ndvi_day': np.nan,
        'max_decline_rate_ndvi_day': np.nan,
        'ndvi_range_fit': annual_range,
        'weak_seasonality': annual_range < MIN_RANGE,
        'valid_spring_decline': event is not None,
        'global_peak_doy_qc': int(np.argmax(v)) + 1,
    }
    if event is None:
        return pd.Series(out)
    peak, trough = event
    duration = trough - peak
    daily_change = np.diff(v[peak:trough+1])
    decline = float(v[peak] - v[trough])
    out.update({
        'spring_peak_doy': peak + 1,
        'summer_min_doy': trough + 1,
        'spring_peak_ndvi_fit': float(v[peak]),
        'summer_min_ndvi_fit': float(v[trough]),
        'decline_duration_days': duration,
        'decline_magnitude_ndvi': decline,
        'mean_decline_rate_ndvi_day': decline / duration,
        'max_decline_rate_ndvi_day': float(-daily_change.min()),
    })
    return pd.Series(out)


valid_sample = sampled.loc[valid].copy().reset_index(drop=True)
metrics = valid_sample.apply(phenology_metrics, axis=1)
plot_features = pd.concat([valid_sample, metrics], axis=1)
fields = ['spring_peak_doy', 'summer_min_doy', 'decline_duration_days',
          'decline_magnitude_ndvi', 'mean_decline_rate_ndvi_day',
          'max_decline_rate_ndvi_day']
print(f'Plot-years: {len(plot_features)}; qualifying spring decline: '
      f'{plot_features.valid_spring_decline.sum()} '
      f'({plot_features.valid_spring_decline.mean():.1%})')
display(plot_features.groupby(YEAR_COL)[fields].agg(['count','median']))
display(plot_features[[PLOT_ID_COL, YEAR_COL, 'valid_spring_decline', *fields]].head())


In [ ]:
# Seasonal reconstructions with explicitly selected peak and subsequent trough.
# Include valid and unqualified curves so selection failures remain visible.
sub = plot_features.sample(n=min(12, len(plot_features)), random_state=42)
fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharex=True)
for ax, (_, rec) in zip(axes.flat, sub.iterrows()):
    v = reconstruct_ndvi(rec)
    ax.plot(DOY, v, color='steelblue', lw=2)
    ax.axvspan(*SPRING_PEAK_WINDOW, color='gold', alpha=0.09)
    if rec.valid_spring_decline:
        p, m = int(rec.spring_peak_doy), int(rec.summer_min_doy)
        ax.scatter([p], [v[p-1]], color='darkorange', s=38, zorder=3)
        ax.scatter([m], [v[m-1]], color='firebrick', s=38, zorder=3)
        ax.axvline(p, color='darkorange', lw=1, ls='--', alpha=.75)
        ax.axvline(m, color='firebrick', lw=1, ls='--', alpha=.75)
        subtitle = f'peak {p} → min {m}'
    else:
        subtitle = 'no qualifying spring decline'
    ax.set_title(f'{rec[PLOT_ID_COL]} · {int(rec[YEAR_COL])} | {subtitle}', fontsize=9)
    ax.set_xlim(1, 365)
    ax.grid(alpha=.15)
for ax in axes.flat[len(sub):]:
    ax.axis('off')
fig.suptitle('K2 NDVI at field plots — first qualified spring max (orange) → next min (red)', fontsize=13)
fig.supxlabel('Day of year')
fig.supylabel('Fitted NDVI')
plt.tight_layout()
plt.show()


In [ ]:
# Distributions of spring maximum, subsequent minimum, and decline behavior.
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, fields):
    x = plot_features.loc[plot_features.valid_spring_decline, col].dropna()
    ax.hist(x, bins=30, edgecolor='white', color='steelblue')
    ax.set_title(col.replace('_', ' '))
    ax.set_ylabel('Plot-year count')
plt.tight_layout()
plt.show()

# QC: how often did the first spring maximum differ from the global maximum?
qualified = plot_features.loc[plot_features.valid_spring_decline]
print('Spring peak differs from global peak:',
      f'{(qualified.spring_peak_doy != qualified.global_peak_doy_qc).mean():.1%}'
      if len(qualified) else 'No qualified plots')

# Repeated plot locations in consecutive observed years (if any).
repeat = plot_features.sort_values([PLOT_ID_COL, YEAR_COL]).copy()
repeat['previous_year'] = repeat.groupby(PLOT_ID_COL)[YEAR_COL].shift()
repeat['previous_spring_peak'] = repeat.groupby(PLOT_ID_COL)['spring_peak_doy'].shift()
repeat = repeat.loc[(repeat[YEAR_COL] - repeat.previous_year) == 1].copy()
repeat['spring_peak_shift_days'] = repeat.spring_peak_doy - repeat.previous_spring_peak
print('Consecutive plot-year pairs:', len(repeat),
      '| with valid peaks:', repeat.spring_peak_shift_days.notna().sum())
if repeat.spring_peak_shift_days.notna().any():
    display(repeat.spring_peak_shift_days.describe())
    plt.figure(figsize=(8, 3))
    plt.hist(repeat.spring_peak_shift_days.dropna(), bins=30, color='steelblue')
    plt.title('Interannual spring-peak DOY difference (same plot, consecutive years)')
    plt.xlabel('Difference in days')
    plt.ylabel('Pairs')
    plt.show()


## Interpretation, limitations, and QC

- **First qualifying spring maximum:** The first local NDVI maximum in DOY 30–180 whose decline to the **next local minimum** is at least 10% of the reconstructed annual NDVI range. The next minimum must be before DOY 300 and within 180 days. These are *exploratory, prespecified algorithm thresholds*, not validated ecological phenophase definitions.
- **No qualifying event:** Metrics are `NaN`, not forced from the global maximum or a year-boundary extremum. Inspect the fraction excluded and how exclusion differs among vegetation states and years before using these features in ML.
- **Decline duration is NOT growing-season length.** It measures days between the selected first spring maximum and subsequent minimum. Rates are calculated from the fitted curve, not raw observations.
- **Phase convention:** Confirm `PHASE_CONVENTION`, units, and time origin against the Earth Engine export by comparing reconstructed trajectories with *observed NDVI* at known plots. Coherent-looking curves alone are insufficient.
- **Spatial support:** Point sampling of the TIFF may not match buffer-aggregated training features. This matters if derived features are later joined to classifier rows.
- **Sampling:** The only pixels sampled are field-plot coordinates. No raster outputs or files are written.
- **2026:** Included in `YEARS` and processed when the matching raster and plot table entries exist.
